# Phase 5 — Full Evaluation Metrics

Goes beyond Phase 4's accuracy/F1/ROC-AUC snapshot: adds specificity, PR-AUC, balanced accuracy, calibration, and Brier score, plus 95% confidence intervals via a **patient-level block bootstrap** (resampling unique `Patient File No.` values, not individual rows — consistent with the patient-as-unit-of-independence rule used everywhere else in this project).

All of this runs on **out-of-fold (OOF) predictions**: every row of `train_pool` is predicted exactly once, by whichever of the 5 CV folds held it out during training. That means every plot here is leakage-free without touching `holdout_validation`, which stays frozen.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import INTERIM_DIR, PATIENT_ID_COL, TABLES_DIR
from src.preprocessing import split_features_target
from src.train import run_all_models_cv_with_oof
from src.evaluate import (
    build_metrics_table_with_ci, get_confusion_matrix, get_roc_curve,
    get_pr_curve, get_calibration_curve,
)
from src.viz import apply_chart_style, save_fig, INK_PRIMARY, INK_SECONDARY, GRIDLINE

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)

train_pool = pd.read_csv(INTERIM_DIR / "train_pool.csv")
X, y = split_features_target(train_pool)
groups = train_pool[PATIENT_ID_COL]

metrics_df, oof_df = run_all_models_cv_with_oof(X, y, groups)
oof_df.to_csv(TABLES_DIR / "phase5_oof_predictions.csv", index=False)
print("OOF predictions:", oof_df.shape, "| one row per (model, train_pool row)")

## Metric comparison table (with 95% bootstrap CI)

In [ ]:
metrics_table = build_metrics_table_with_ci(oof_df, n_boot=1000)
metrics_table.to_csv(TABLES_DIR / "phase5_metrics_with_ci.csv")
metrics_table[["accuracy", "recall_sensitivity", "specificity", "f1", "roc_auc", "pr_auc", "balanced_accuracy", "brier_score"]].round(3)

The 5 real models are compared in the curve plots below; Dummy baseline is excluded from them (its ROC/PR curves are, by construction, a single point on the chance line and add nothing visually) but stays in the table above as the floor reference.

In [ ]:
MODEL_ORDER = ["logistic_regression", "decision_tree", "random_forest", "gradient_boosting", "xgboost"]
ALL_MODEL_ORDER = ["dummy_baseline"] + MODEL_ORDER
MODEL_LABELS = {
    "logistic_regression": "Logistic Regression", "decision_tree": "Decision Tree",
    "random_forest": "Random Forest", "gradient_boosting": "Gradient Boosting",
    "xgboost": "XGBoost", "dummy_baseline": "Dummy baseline",
}
MODEL_COLORS = {
    "logistic_regression": "#2a78d6", "decision_tree": "#eb6834",
    "random_forest": "#1baf7a", "gradient_boosting": "#eda100", "xgboost": "#e87ba4",
}

## Confusion matrices (out-of-fold, all 6 models)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, model_name in zip(axes.flat, ALL_MODEL_ORDER):
    sub = oof_df[oof_df["model"] == model_name]
    cm = get_confusion_matrix(sub)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            color = "white" if cm_norm[i, j] > 0.6 else INK_PRIMARY
            ax.text(j, i, f"{cm[i,j]}\n({cm_norm[i,j]:.0%})", ha="center", va="center", fontsize=9.5, color=color)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["No PCOS", "PCOS"], fontsize=8.5)
    ax.set_yticks([0, 1]); ax.set_yticklabels(["No PCOS", "PCOS"], fontsize=8.5)
    ax.set_xlabel("Predicted", fontsize=8.5, color=INK_SECONDARY)
    ax.set_ylabel("Actual", fontsize=8.5, color=INK_SECONDARY)
    ax.set_title(MODEL_LABELS[model_name], fontsize=10, color=INK_PRIMARY)
    for spine in ax.spines.values():
        spine.set_visible(False)
fig.suptitle("Confusion matrices — out-of-fold predictions on train_pool", fontsize=11)
save_fig(fig, "13_confusion_matrices", tight_layout_rect=(0, 0, 1, 0.95))
plt.show()

**Interpretation:** Random Forest makes the fewest false positives (66, 5% of No-PCOS patients misclassified) but the most false negatives among the ensembles (140, 23% of true PCOS patients missed). Gradient Boosting and Logistic Regression catch more true PCOS cases (117 and 108 missed respectively) at the cost of a few more false alarms. Decision Tree is worst on both sides simultaneously (233 false negatives, 252 false positives) — consistent with its overfitting.

**Why this matters for a screening tool:** a false negative here means a patient with PCOS is told their estimated risk is low — the costlier error in a screening context. Random Forest's precision/specificity advantage comes specifically at the expense of the recall this project's own design brief says should be prioritized.

## ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
ax.plot([0, 1], [0, 1], linestyle="--", color=GRIDLINE, linewidth=1.5, label="Chance (AUC=0.50)", zorder=1)
for model_name in MODEL_ORDER:
    sub = oof_df[oof_df["model"] == model_name]
    fpr, tpr, _ = get_roc_curve(sub)
    auc = metrics_table.loc[model_name, "roc_auc"]
    ax.plot(fpr, tpr, color=MODEL_COLORS[model_name], linewidth=2, label=f"{MODEL_LABELS[model_name]} (AUC={auc:.3f})")
ax.set_xlabel("False positive rate (1 − specificity)")
ax.set_ylabel("True positive rate (recall)")
ax.set_title("ROC curves — out-of-fold predictions")
ax.legend(frameon=False, fontsize=8, loc="lower right")
apply_chart_style(ax)
save_fig(fig, "14_roc_curves")
plt.show()

**Interpretation:** Random Forest (0.953), Gradient Boosting (0.944), and XGBoost (0.930) form a tight top cluster; Logistic Regression (0.918) trails them by only a little despite being far simpler. Decision Tree (0.723) is well behind everything else, and its curve's blocky shape (visibly fewer distinct steps) is itself a symptom of being one unpruned tree with a small number of achievable probability thresholds, rather than an averaged ensemble.

## Precision-Recall curves

In [ ]:
prevalence = y.mean()
fig, ax = plt.subplots(figsize=(6, 5.5))
ax.axhline(prevalence, linestyle="--", color=GRIDLINE, linewidth=1.5, label=f"No-skill baseline (prevalence={prevalence:.2f})", zorder=1)
for model_name in MODEL_ORDER:
    sub = oof_df[oof_df["model"] == model_name]
    precision, recall, _ = get_pr_curve(sub)
    pr_auc = metrics_table.loc[model_name, "pr_auc"]
    ax.plot(recall, precision, color=MODEL_COLORS[model_name], linewidth=2, label=f"{MODEL_LABELS[model_name]} (PR-AUC={pr_auc:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall curves — out-of-fold predictions")
ax.legend(frameon=False, fontsize=8, loc="lower left")
apply_chart_style(ax)
save_fig(fig, "15_pr_curves")
plt.show()

**Interpretation:** with 31% PCOS prevalence in `train_pool`, PR-AUC is the harsher, more informative metric of the two rank metrics (ROC-AUC can look optimistic under imbalance). Random Forest (0.907) and Gradient Boosting (0.900) again lead; Logistic Regression (0.828) holds up reasonably; Decision Tree (0.494) is barely better than the 0.31 no-skill baseline at high recall.

## Calibration curves

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
ax.plot([0, 1], [0, 1], linestyle="--", color=GRIDLINE, linewidth=1.5, label="Perfectly calibrated", zorder=1)
for model_name in MODEL_ORDER:
    sub = oof_df[oof_df["model"] == model_name]
    frac_pos, mean_pred = get_calibration_curve(sub, n_bins=10)
    ax.plot(mean_pred, frac_pos, marker="o", markersize=4, color=MODEL_COLORS[model_name], linewidth=1.8, label=MODEL_LABELS[model_name])
ax.set_xlabel("Mean predicted probability (per bin)")
ax.set_ylabel("Observed PCOS rate (per bin)")
ax.set_title("Calibration curves — out-of-fold predictions (10 quantile bins)")
ax.legend(frameon=False, fontsize=8, loc="upper left")
apply_chart_style(ax)
save_fig(fig, "16_calibration_curves")
plt.show()

**Interpretation — the one real surprise in this phase:** Random Forest has the best ROC-AUC and PR-AUC, but its calibration curve is the worst of the five — a pronounced S-shape, pushing mid-range predictions toward 0 or 1 rather than tracking the diagonal. This is typical of averaged-tree-vote probabilities and means Random Forest's predicted *probabilities* are less trustworthy as literal percentages than its ranking ability would suggest. Gradient Boosting tracks the diagonal most closely through the low-to-mid range; Logistic Regression is reasonable but underconfident in the 0.2-0.5 range.

**Why this matters for this specific project:** the planned app (Phase 10) shows the user a literal number — "Estimated PCOS risk: 81%". If the final model is Random Forest, that number should not be read as a calibrated probability without recalibration (e.g. `CalibratedClassifierCV` / Platt scaling) — a decision to make explicitly in Phase 6 if Random Forest is carried forward, rather than silently shipping a misleading percentage.

**Limitation:** Brier score (table above) doesn't fully capture this — it blends calibration and discrimination together, and Random Forest's Brier score (0.087) is still the best of the five because its strong discrimination compensates numerically. The calibration curve is telling a story the Brier score alone would hide.

## Tradeoffs among models — summary for model selection (Phase 6)

| Model | Strongest at | Weakest at | Note |
|---|---|---|---|
| **Random Forest** | ROC-AUC (0.953), PR-AUC (0.907), specificity (0.953), precision (0.879) | Recall (0.775 — lowest of the real models except Decision Tree); calibration (visibly S-shaped) | Best ranking ability, worst-calibrated probabilities, fewest false positives but most false negatives among the ensembles |
| **Gradient Boosting** | Recall (0.812) and F1 (0.806) among ensembles; best-tracking calibration curve | Slightly behind Random Forest on ROC-AUC/PR-AUC | Best overall balance if calibrated probabilities and recall both matter |
| **Logistic Regression** | Highest recall of any real model (0.826); smallest train-val overfitting gap (Phase 4); fully interpretable coefficients | ROC-AUC/PR-AUC trail the ensembles by a modest margin | Strongest choice if interpretability or recall is prioritized over the last few points of ranking performance |
| **XGBoost** | Comparable to Gradient Boosting, slightly behind on every metric measured | No clear advantage found here over Gradient Boosting | Not a strong differentiator in this dataset — likely not worth the added complexity over Gradient Boosting |
| **Decision Tree** | Nothing | Worst on every real metric | Kept only to illustrate overfitting (Phase 4); not a serious candidate |

**Given this project's own stated priority — recall matters because false negatives mean missed PCOS cases — Gradient Boosting and Logistic Regression currently make a stronger case than Random Forest's leaderboard-topping ROC-AUC alone would suggest.** Nothing is finalized here: Phase 6 tunes the promising candidates (likely Random Forest, Gradient Boosting, Logistic Regression) and Phase 7 checks whichever is selected against `holdout_validation` before anything is frozen.